In [32]:
import os
from dotenv import load_dotenv  

# Configure environment variables  
load_dotenv()  

DEPLOYMENT_ID = os.getenv("AZURE_OPENAI_DEPLOYED_MODEL") 
OPENAI_API_BASE = os.getenv("AZURE_OPENAI_API_BASE") 
OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION") 
OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

In [33]:
import openai
import json

# Load config values
# with open(r'config.json') as config_file:
    # config_details = json.load(config_file)
    
# Setting up the deployment name
# deployment_id = config_details['DEPLOYMENT_ID']
deployment_id = DEPLOYMENT_ID
# if using openai
# deployment_id = 'gpt-3.5-turbo-0613'

# This is set to `azure`
openai.api_type = "azure"
# openai.api_type = "open_ai"

# The API key for your Azure OpenAI resource.
openai.api_key = OPENAI_API_KEY

# The base URL for your Azure OpenAI resource. e.g. "https://<your resource name>.openai.azure.com"
openai.api_base = OPENAI_API_BASE

# Currently Chat Completion API have the following versions available: 2023-07-01-preview
openai.api_version = OPENAI_API_VERSION

In [41]:
print(openai.api_version)
print(openai.api_base)
print(openai.api_type)
print(deployment_id)
print(openai.VERSION)

2023-07-01-preview
https://openai-skyblue-east.openai.azure.com/
azure
gpt-35-turbo-0613
0.27.8


## 1.0 Test functions
This code calls the model with the user query and the set of functions defined in the functions parameter. 

The model then can choose if it calls a function. If a function is called, the content will be in a stringified JSON object. The function call that should be made and arguments are located in

`response[choices][0][function_call]`

In [46]:
def get_function_call(messages, function_call = "auto"):
    # Define the functions to use
    functions = [
        {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    ]

    # Call the model with the user query (messages) and the functions defined in the functions parameter
    response = openai.ChatCompletion.create(
        deployment_id = deployment_id,
        messages=messages,
        functions=functions,
        function_call=function_call, 
    )

    return response

In [44]:
response = openai.ChatCompletion.create(
        # model = deployment_id,
        # engine = deployment_id,
        deployment_id = deployment_id,
        messages=[{"role": "user", "content": "What's the weather like in San Francisco?"}],
        functions=[
        {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    ],
        function_call='auto', 
    )   

In [45]:
response

<OpenAIObject chat.completion id=chatcmpl-7hNxsrT55ffjxZu5uaoIobsFMVCPO at 0x7f3f346ab010> JSON: {
  "id": "chatcmpl-7hNxsrT55ffjxZu5uaoIobsFMVCPO",
  "object": "chat.completion",
  "created": 1690574580,
  "model": "gpt-35-turbo",
  "prompt_annotations": [
    {
      "prompt_index": 0,
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      }
    }
  ],
  "choices": [
    {
      "index": 0,
      "finish_reason": "function_call",
      "message": {
        "role": "assistant",
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\n  \"location\": \"San Francisco\"\n}"
        }
      },
      "content_filter

In [47]:
# help(openai.ChatCompletion.create)

### Forcing the use of a specific function or no function
By changing the value of the `functions` parameter you can allow the model to decide what function to use, force the model to use a specific function, or force the model to use no function.

In [48]:
first_message = [{"role": "user", "content": "What's the weather like in San Francisco?"}]
# 'auto' : Let the model decide what function to call
print("Let the model decide what function to call:")
print (get_function_call(first_message, "auto"),'\n\n')

# 'none' : Don't call any function 
print("Don't call any function:")
print (get_function_call(first_message, "none"),'\n\n')

# force a specific function call
print("Force a specific function call:")
print (get_function_call(first_message, function_call={"name": "get_current_weather"}))

Let the model decide what function to call:
{
  "id": "chatcmpl-7hNyg2wdjbpgdMQQqxCLKJh2SkSrR",
  "object": "chat.completion",
  "created": 1690574630,
  "model": "gpt-35-turbo",
  "prompt_annotations": [
    {
      "prompt_index": 0,
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      }
    }
  ],
  "choices": [
    {
      "index": 0,
      "finish_reason": "function_call",
      "message": {
        "role": "assistant",
        "function_call": {
          "name": "get_current_weather",
          "arguments": "{\n\"location\": \"San Francisco, CA\"\n}"
        }
      },
      "content_filter_results": {}
    }
  ],
  "usage": {
    "completi

RateLimitError: Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2023-07-01-preview have exceeded call rate limit of your current OpenAI S0 pricing tier. Please retry after 10 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit.